In [1]:
import os
import sys
from pathlib import Path
import logging
import time

from websocket_server import WebsocketServer
from multiprocessing import Process

import pickle
import json
import pandas as pd
import numpy as np

cur_dir = os.getcwd()
path = Path(cur_dir)
sys.path.insert(0, str(path.parent.absolute()))

from src.nre.preprocess import preprocess_df
from src.nre.real_time_model import NetworkModel
from src.nre.time_windowed import get_window
from src.nre.safe_routing import communication_graph_from_df
from src.nre.stream_data import start_web_socket_server, start_stream

In [2]:
with open(r'saves\victim_net.pickle', 'rb') as handle:
   names = pickle.load(handle) 

len(names)

13

In [3]:
with open(r'saves\connected_84.pickle', 'rb') as handle:
   names = pickle.load(handle) 

len(names)

84

In [4]:


df_raw = pd.read_csv('..\CIC-IDS-2017\GeneratedLabelledFlows\TrafficLabelling\Tuesday-WorkingHours.pcap_ISCX.csv' , header=0, encoding='cp1252')
df = preprocess_df(df_raw, date_col=' Timestamp')

idx = df[' Source IP'].isin(names) & df[' Destination IP'].isin(names)
df = df[idx].copy()
df_temp = df.iloc[:10000,:]

In [5]:
df_temp

,Flow ID,Source IP,Source Port,Destination IP,Destination Port,Protocol,Timestamp,Flow Duration,Total Fwd Packets,Total Backward Packets,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
133451,192.168.10.16-192.168.10.50-137-137-17,192.168.10.50,137,192.168.10.16,137,17,2017-04-07 01:00:00,47,2,0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
133452,192.168.10.12-192.168.10.50-137-137-17,192.168.10.50,137,192.168.10.12,137,17,2017-04-07 01:00:00,3,2,0,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
334175,192.168.10.50-192.168.10.51-21-53143-6,192.168.10.51,53143,192.168.10.50,21,6,2017-04-07 01:00:00,51,1,2,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
133516,192.168.10.16-192.168.10.50-58110-139-6,192.168.10.16,58110,192.168.10.50,139,6,2017-04-07 01:00:00,95,3,1,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
404986,192.168.10.3-192.168.10.5-445-56898-6,192.168.10.5,56898,192.168.10.3,445,6,2017-04-07 01:00:00,91534888,28,24,...,20,15644.0,31186.66682,62424.0,48.0,22900000.0,9215142.945,30000000.0,10700000.0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
393898,192.168.10.3-192.168.10.51-53-9950-17,192.168.10.51,9950,192.168.10.3,53,17,2017-04-07 01:51:00,159,2,2,...,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
332328,192.168.10.3-192.168.10.17-53-10964-17,192.168.10.17,10964,192.168.10.3,53,17,2017-04-07 01:51:00,174,2,2,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
153197,172.217.10.142-192.168.10.16-443-44558-6,192.168.10.16,44558,172.217.10.142,443,6,2017-04-07 01:51:00,293622,12,11,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
429685,192.168.10.3-192.168.10.17-53-48661-17,192.168.10.17,48661,192.168.10.3,53,17,2017-04-07 01:51:00,23980,2,2,...,32,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [6]:
import pickle

with open('saves/stream_data_84.pickle', 'wb') as file:
    pickle.dump(df, file)

In [7]:
with open('saves/stream_data_84.pickle', 'rb') as file:
    df = pickle.load(file)

In [7]:
import src.nre.network_connectivity, importlib

importlib.reload(src.nre.network_connectivity)
from src.nre.network_connectivity import*

In [8]:
import src.nre.real_time_model, importlib

importlib.reload(src.nre.real_time_model)
from src.nre.real_time_model import*

In [8]:
import src.nre.stream_data, importlib

importlib.reload(src.nre.stream_data)
from src.nre.stream_data import start_web_socket_server, start_stream

In [9]:
server = start_web_socket_server()

INFO:websocket_server.websocket_server:Listening on port 15674 for clients..
INFO:websocket_server.websocket_server:Starting WebsocketServer on thread Thread-5 (serve_forever).


Starting AMQP Server


In [ ]:
start_stream(server, df, names, grow_entities=False, window_type='conn', graph_conn_size=500, conn_size = 100)#, save_mode='single')

Waiting for Client
Client Connected
Graph #1
Conditioning number:  1.980447838578323e+17 
Determinant of F^T*F:  8.644933756369339e-176
Graph #2
Conditioning number:  9.009616146726985e+17 
Determinant of F^T*F:  2.7425652578722357e-162


INFO:websocket_server.websocket_server:Client asked to close connection.


Graph #3
Conditioning number:  3.6028362782176115e+17 
Determinant of F^T*F:  6.479104989531761e-142
Waiting for Client
Client Connected
Graph #1
Conditioning number:  1.980447838578323e+17 
Determinant of F^T*F:  8.644933756369339e-176


INFO:websocket_server.websocket_server:Client asked to close connection.


Graph #2
Conditioning number:  9.009616146726985e+17 
Determinant of F^T*F:  2.7425652578722357e-162
Waiting for Client
Client Connected
Graph #1
Conditioning number:  1.980447838578323e+17 
Determinant of F^T*F:  8.644933756369339e-176
Graph #2
Conditioning number:  9.009616146726985e+17 
Determinant of F^T*F:  2.7425652578722357e-162
Graph #3
Conditioning number:  3.6028362782176115e+17 
Determinant of F^T*F:  6.479104989531761e-142
Graph #4
Conditioning number:  5.802411952860541e+16 
Determinant of F^T*F:  -4.28565535541971e-157
Graph #5
Conditioning number:  9.070656369802597e+17 
Determinant of F^T*F:  0.0
Graph #6
Conditioning number:  4.083346016818143e+17 
Determinant of F^T*F:  -1.8123702047095284e-218
Graph #7
Conditioning number:  1.0786732904930728e+18 
Determinant of F^T*F:  -4.263338277352647e-143
Graph #8
Conditioning number:  591.1091308086962 
Determinant of F^T*F:  2.986678703418434e-110
Graph #9
Conditioning number:  3.733023982311929e+16 
Determinant of F^T*F:  -2.

INFO:websocket_server.websocket_server:Client asked to close connection.


Graph #25
Conditioning number:  17376.445839950302 
Determinant of F^T*F:  -8.153588799903567e-143
Waiting for Client


In [8]:
start_stream(server, df, names, grow_entities=True, window_type='conn', graph_conn_size=1500, conn_size = 100)

Waiting for Client
Client Connected
Graph #1
Conditioning number:  1.0780719062004976e+17 
Determinant of F^T*F:  0.0
{"names": ["192.168.10.50", "192.168.10.16", "192.168.10.12", "192.168.10.51", "192.168.10.5", "192.168.10.3", "192.168.10.8", "192.168.10.25", "192.168.10.19", "172.217.12.206", "192.168.10.9", "192.168.10.15", "38.69.238.16", "192.168.10.17", "172.217.12.174", "192.168.10.14", "172.217.12.132", "199.96.57.6", "172.217.3.98", "52.40.24.51", "178.255.83.1", "50.31.164.173", "172.217.12.134", "69.172.216.56", "157.240.18.35", "104.97.73.166", "138.108.7.20"], "funcEdges": [[1.0, 0.218718777479467, 0.20221347584160188, 0.18184164086108398, 0.2660932328263378, 0.2722655544456344, 0.3314570206621452, 0.42282991302586187, 0.10286205973767161, 0.3453488510088891, 0.11474159364247995, 0.10769364017510291, 0.26609323282633784, 0.20739528911052146, 0.1443132188401811, 0.09521458867706131, 0.08001451095630384, 0.13290927123169302, 0.12392496354385601, 0.14175313976258835, 0.18292

Graph #2
Conditioning number:  4.124266737274775e+17 
Determinant of F^T*F:  -6.526927811128413e-41
{"names": ["192.168.10.50", "192.168.10.16", "192.168.10.12", "192.168.10.51", "192.168.10.5", "192.168.10.3", "192.168.10.8", "192.168.10.25", "192.168.10.19", "172.217.12.206", "192.168.10.9", "192.168.10.15", "38.69.238.16", "192.168.10.17", "172.217.12.174", "192.168.10.14", "172.217.12.132", "199.96.57.6", "172.217.3.98", "52.40.24.51", "178.255.83.1", "50.31.164.173", "172.217.12.134", "69.172.216.56", "157.240.18.35", "104.97.73.166", "138.108.7.20", "54.152.24.232", "68.67.178.109", "172.217.7.6", "72.30.202.150", "152.163.66.165", "68.67.180.12", "68.67.178.199", "52.84.145.87"], "funcEdges": [[1.0, 0.13863478251851846, 0.2335215265518315, 0.10066379831630076, 0.41427820304033086, 0.19704825355286923, 0.6015178069628423, 0.4556640605384116, 0.08568855476068306, 0.1949325835930478, 0.24573991648533672, 0.10969427075604182, 0.13304661641316892, 0.10383968792748983, 0.0784675353696

Graph #3
Conditioning number:  4470.337824798219 
Determinant of F^T*F:  1.1535761331478514e-07
{"names": ["192.168.10.50", "192.168.10.16", "192.168.10.12", "192.168.10.51", "192.168.10.5", "192.168.10.3", "192.168.10.8", "192.168.10.25", "192.168.10.19", "172.217.12.206", "192.168.10.9", "192.168.10.15", "38.69.238.16", "192.168.10.17", "172.217.12.174", "192.168.10.14", "172.217.12.132", "199.96.57.6", "172.217.3.98", "52.40.24.51", "178.255.83.1", "50.31.164.173", "172.217.12.134", "69.172.216.56", "157.240.18.35", "104.97.73.166", "138.108.7.20", "54.152.24.232", "68.67.178.109", "172.217.7.6", "72.30.202.150", "152.163.66.165", "68.67.180.12", "68.67.178.199", "52.84.145.87", "23.15.4.16", "172.217.11.34", "31.13.71.36", "8.43.72.98"], "funcEdges": [[1.0, 0.22693284952612122, 0.4922419766725109, 0.11057474153021687, 0.37969944766452357, 0.23600362274772418, 0.5284131680202374, 0.48815004592716615, 0.25441543198089944, 0.28766416173187803, 0.17210689631589113, 0.09651008385686385,

Graph #4
Conditioning number:  3009.813075367039 
Determinant of F^T*F:  3.7185104019598894e-09
{"names": ["192.168.10.50", "192.168.10.16", "192.168.10.12", "192.168.10.51", "192.168.10.5", "192.168.10.3", "192.168.10.8", "192.168.10.25", "192.168.10.19", "172.217.12.206", "192.168.10.9", "192.168.10.15", "38.69.238.16", "192.168.10.17", "172.217.12.174", "192.168.10.14", "172.217.12.132", "199.96.57.6", "172.217.3.98", "52.40.24.51", "178.255.83.1", "50.31.164.173", "172.217.12.134", "69.172.216.56", "157.240.18.35", "104.97.73.166", "138.108.7.20", "54.152.24.232", "68.67.178.109", "172.217.7.6", "72.30.202.150", "152.163.66.165", "68.67.180.12", "68.67.178.199", "52.84.145.87", "23.15.4.16", "172.217.11.34", "31.13.71.36", "8.43.72.98", "104.19.195.102", "172.217.10.142", "104.88.70.158", "216.58.217.99"], "funcEdges": [[1.0, 0.20707444753296916, 0.41155503696171075, 0.26789596187957443, 0.4145444023274311, 0.17681519280549832, 0.4639898125975135, 0.47082005301377694, 0.14680142838

Graph #5
Conditioning number:  735.3535639673521 
Determinant of F^T*F:  4.1784750930949494e-05
{"names": ["192.168.10.50", "192.168.10.16", "192.168.10.12", "192.168.10.51", "192.168.10.5", "192.168.10.3", "192.168.10.8", "192.168.10.25", "192.168.10.19", "172.217.12.206", "192.168.10.9", "192.168.10.15", "38.69.238.16", "192.168.10.17", "172.217.12.174", "192.168.10.14", "172.217.12.132", "199.96.57.6", "172.217.3.98", "52.40.24.51", "178.255.83.1", "50.31.164.173", "172.217.12.134", "69.172.216.56", "157.240.18.35", "104.97.73.166", "138.108.7.20", "54.152.24.232", "68.67.178.109", "172.217.7.6", "72.30.202.150", "152.163.66.165", "68.67.180.12", "68.67.178.199", "52.84.145.87", "23.15.4.16", "172.217.11.34", "31.13.71.36", "8.43.72.98", "104.19.195.102", "172.217.10.142", "104.88.70.158", "216.58.217.99", "74.119.118.82"], "funcEdges": [[1.0, 0.14205887039604054, 0.2501815291463051, 0.35678442468328486, 0.43207539565258707, 0.10565552160524047, 0.3391988534799244, 0.657993644426645

INFO:websocket_server.websocket_server:Client asked to close connection.


Graph #6
Conditioning number:  147767.3154397401 
Determinant of F^T*F:  2.350179953221626e-09
{"names": ["192.168.10.50", "192.168.10.16", "192.168.10.12", "192.168.10.51", "192.168.10.5", "192.168.10.3", "192.168.10.8", "192.168.10.25", "192.168.10.19", "172.217.12.206", "192.168.10.9", "192.168.10.15", "38.69.238.16", "192.168.10.17", "172.217.12.174", "192.168.10.14", "172.217.12.132", "199.96.57.6", "172.217.3.98", "52.40.24.51", "178.255.83.1", "50.31.164.173", "172.217.12.134", "69.172.216.56", "157.240.18.35", "104.97.73.166", "138.108.7.20", "54.152.24.232", "68.67.178.109", "172.217.7.6", "72.30.202.150", "152.163.66.165", "68.67.180.12", "68.67.178.199", "52.84.145.87", "23.15.4.16", "172.217.11.34", "31.13.71.36", "8.43.72.98", "104.19.195.102", "172.217.10.142", "104.88.70.158", "216.58.217.99", "74.119.118.82", "172.217.10.138", "172.217.10.33"], "funcEdges": [[1.0, 0.25474108739678686, 0.5303810724343181, 0.318206232641292, 0.4243858408273442, 0.07850257944025577, 0.3638

KeyboardInterrupt: 

In [44]:
server.clients




[]

In [45]:
server.disconnect_clients_gracefully()

In [46]:
server.shutdown_gracefully()

ERROR:websocket_server.websocket_server:********************************************************************************
Exception in child thread <WebsocketServerThread(Thread-14 (serve_forever), started daemon 36084)>: [WinError 10038] An operation was attempted on something that is not a socket
********************************************************************************
Traceback (most recent call last):
  File "C:\Users\bayer\PycharmProjects\NRE\venv2\lib\site-packages\websocket_server\thread.py", line 27, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\bayer\AppData\Local\Programs\Python\Python310\lib\socketserver.py", line 232, in serve_forever
    ready = selector.select(poll_interval)
  File "C:\Users\bayer\AppData\Local\Programs\Python\Python310\lib\selectors.py", line 324, in select
    r, w, _ = self._select(self._readers, self._writers, [], timeout)
  File "C:\Users\bayer\AppData\Local\Programs\Python\Python310\lib\selectors.py", line 315, in _sele

In [47]:
server.shutdown_abruptly()

In [48]:
del server